# Day 64 - continuous batching and the inference scheduler

A serving engine is a loop that decides, a few hundred times a second, which requests go
into the next forward pass. This notebook simulates that loop: static against continuous
batching, the waiting queue, chunked prefill, a KV pool that runs out, preemption, the
scheduling policies - and measures TTFT, TPOT, ITL, throughput and goodput for each.

## 1. The machine, and what one step costs

The weights are read once per *step*, not once per request, so a request added to a
running batch is nearly free until the KV reads catch up with the weight read.

In [ ]:
from scheduling import (GPU, STEP_WEIGHT_MS, KV_MS_PER_TOKEN, PREFILL_MS_PER_TOKEN,
                        decode_step_ms, prefill_ms, swap_ms, workload,
                        run_static, run_continuous, metrics, goodput,
                        rate_sweep, batching_table, chunk_table, pool_table,
                        policy_table, check)

print(f"weights per step   {STEP_WEIGHT_MS:.2f} ms")
print(f"one KV token       {KV_MS_PER_TOKEN * 1e3:.3f} us")
print(f"one prompt token   {PREFILL_MS_PER_TOKEN:.4f} ms")
print(f"KV pool            {GPU['kv_pool_tokens']:,} tokens\n")
for b, ctx in ((1, 2_000), (16, 32_000), (64, 128_000), (128, 256_000)):
    ms = decode_step_ms(ctx, b)
    print(f"batch {b:4}: {ms:5.2f} ms/step, {b / ms * 1e3:8.0f} tok/s, "
          f"{ms / b * 1e3:6.1f} us per token")

weights per step   3.30 ms
one KV token       0.008 us
one prompt token   0.0268 ms
KV pool            150,000 tokens

batch    1:  3.32 ms/step,      301 tok/s, 3317.0 us per token
batch   16:  3.54 ms/step,     4515 tok/s,  221.5 us per token
batch   64:  4.27 ms/step,    14996 tok/s,   66.7 us per token
batch  128:  5.23 ms/step,    24456 tok/s,   40.9 us per token


## 2. Static against continuous batching

Same requests, same hardware. A static batch runs until its longest member finishes, so
slots sit occupied by requests that are already done, and arrivals wait for the edge of
the batch.

In [ ]:
rows, w = batching_table()
for r in rows:
    print(f"{r['name']:28} TTFT p50 {r['ttft_p50']:6.2f} s   "
          f"{r['throughput']:6.0f} tok/s   padding {r['pad_waste']:4.0%}   "
          f"done in {r['end_s']:4.0f} s")

static batching (B=8)        TTFT p50  25.62 s      807 tok/s   padding  56%   done in   81 s
static batching (B=32)       TTFT p50   9.70 s     1368 tok/s   padding  72%   done in   48 s
continuous batching          TTFT p50   0.06 s     2140 tok/s   padding   0%   done in   30 s


## 3. The latency-throughput curve

Below the knee the engine is arrival-limited; above it the queue grows and only the
throughput column still looks healthy.

In [ ]:
for r in rate_sweep():
    print(f"{r['rate']:3}/s   {r['throughput']:6.0f} tok/s   TTFT p50 {r['ttft_p50']:6.2f} s   "
          f"p99 {r['ttft_p99']:6.2f} s   ITL p99 {r['itl_p99']:7.1f} ms   "
          f"goodput {r['goodput']:5.0%}")

  4/s      743 tok/s   TTFT p50   0.03 s   p99   0.66 s   ITL p99    20.3 ms   goodput   91%
  8/s     1463 tok/s   TTFT p50   0.04 s   p99   0.79 s   ITL p99    43.8 ms   goodput   82%
 12/s     2140 tok/s   TTFT p50   0.06 s   p99   1.04 s   ITL p99    61.2 ms   goodput   40%
 16/s     2191 tok/s   TTFT p50   2.43 s   p99   6.63 s   ITL p99   431.2 ms   goodput    1%
 20/s     2384 tok/s   TTFT p50   2.96 s   p99   7.88 s   ITL p99   431.1 ms   goodput    1%
 24/s     2258 tok/s   TTFT p50   4.82 s   p99  11.89 s   ITL p99   431.2 ms   goodput    1%


## 4. Chunked prefill

One long prompt run as a single step is a gap of that length for every request that is
decoding. Cutting it into chunks moves the tail from the prompt size to the chunk size,
and pushes the first token out a little.

In [ ]:
for r in chunk_table(w):
    label = 'whole prompt' if not r['chunk'] else f"{r['chunk']} tokens"
    print(f"{label:14} TTFT p50 {r['ttft_p50']:5.2f} s   ITL p50 {r['itl_p50']:5.1f} ms   "
          f"ITL p99 {r['itl_p99']:6.1f} ms   {r['throughput']:6.0f} tok/s")

whole prompt   TTFT p50  0.06 s   ITL p50   3.6 ms   ITL p99   61.2 ms     2140 tok/s
1024 tokens    TTFT p50  0.11 s   ITL p50   3.7 ms   ITL p99   31.7 ms     2140 tok/s
512 tokens     TTFT p50  0.24 s   ITL p50   3.7 ms   ITL p99   17.9 ms     2140 tok/s
256 tokens     TTFT p50  0.43 s   ITL p50  10.4 ms   ITL p99   10.8 ms     2139 tok/s


## 5. The KV pool, preemption and admission control

Admitting a request promises it cache space for tokens nobody has generated yet. When
the pool fills, somebody is evicted - and recompute throws the progress away.

In [ ]:
for r in pool_table(w):
    cap = r['cap'] or 'none'
    print(f"pool {r['pool']:7,}  cap {str(cap):>4}  preemptions {r['preemptions']:3}  "
          f"{r['throughput']:6.0f} tok/s  ITL p99 {r['itl_p99']:6.1f} ms  "
          f"goodput {r['goodput']:5.0%}")
sw = run_continuous(w, pool_tokens=60_000, preempt='swap')
m = metrics(sw)
print(f"pool  60,000  swap instead of recompute: preemptions {m['preemptions']}, "
      f"{m['throughput']:.0f} tok/s, goodput {goodput(sw):.0%}")
print(f"(swapping a 4,000-token context costs {swap_ms(4000):.0f} ms over PCIe, round trip)")

pool 150,000  cap none  preemptions   0    2140 tok/s  ITL p99   61.2 ms  goodput   40%
pool 150,000  cap   24  preemptions   0    2140 tok/s  ITL p99   75.4 ms  goodput   29%
pool 150,000  cap   16  preemptions   0    1985 tok/s  ITL p99   62.0 ms  goodput   25%
pool  60,000  cap none  preemptions  59    1238 tok/s  ITL p99  428.4 ms  goodput    4%
pool  60,000  cap   24  preemptions   7    2013 tok/s  ITL p99  104.4 ms  goodput   12%
pool  60,000  cap   16  preemptions   0    1985 tok/s  ITL p99   59.5 ms  goodput   26%
pool  60,000  swap instead of recompute: preemptions 16, 2140 tok/s, goodput 29%
(swapping a 4,000-token context costs 5 ms over PCIe, round trip)


## 6. Which request goes first

Shortest-job-first halves the mean and doubles the latency of the requests that were
already slowest.

In [ ]:
for r in policy_table():
    print(f"{r['policy']:5}  mean {r['mean']:5.1f} s  p50 {r['p50']:5.1f}  p99 {r['p99']:5.1f}  "
          f"max {r['max']:5.1f}  long prompts {r['long_mean']:5.1f} s  "
          f"{r['throughput']:6.0f} tok/s")

fcfs   mean   5.8 s  p50   5.5  p99  12.5  max  15.7  long prompts   5.6 s    2550 tok/s
sjf    mean   3.1 s  p50   2.0  p99  19.9  max  21.7  long prompts  10.0 s    2579 tok/s


## 7. Goodput, not throughput

Three configurations at the same throughput, and the number that separates them.

In [ ]:
cfgs = [('continuous, no limits', run_continuous(w)),
        ('+ chunked prefill (512)', run_continuous(w, chunk=512)),
        ('+ admission cap (32)', run_continuous(w, chunk=512, admit_cap=32)),
        ('small pool, no cap', run_continuous(w, pool_tokens=60_000))]
for name, run in cfgs:
    m = metrics(run)
    print(f"{name:24} {m['throughput']:6.0f} tok/s   TTFT p99 {m['ttft_p99']:6.2f} s   "
          f"ITL p99 {m['itl_p99']:6.1f} ms   goodput {goodput(run):5.0%}")

continuous, no limits      2140 tok/s   TTFT p99   1.04 s   ITL p99   61.2 ms   goodput   40%
+ chunked prefill (512)    2140 tok/s   TTFT p99   1.60 s   ITL p99   17.9 ms   goodput  100%
+ admission cap (32)       2140 tok/s   TTFT p99   1.60 s   ITL p99   17.8 ms   goodput  100%
small pool, no cap         1238 tok/s   TTFT p99  22.65 s   ITL p99  428.4 ms   goodput    4%


## 8. Assertions

In [ ]:
check()
print('all assertions passed')

all assertions passed
